# PCA + per-stratum outlier detection — PD vs HC (n=155)

Runs PCA + per-`cell_type` robust Mahalanobis outlier detection on both
VST matrices (class-aware and stratum-aware) for the PD vs HC cohort.

Per-stratum (cell_type) MinCovDet is used because sorted-cell RNA-seq has
biologically distinct PBMC / CD4 / CD8 sub-populations with different mean
expression profiles — a global Mahalanobis would pool them and falsely flag
perfectly-typical samples for their own cell_type as outliers.

In [5]:
"""
PCA + per-stratum outlier detection for VST-normalised RNA-seq data
===================================================================

For each VST matrix (class-aware and stratum-aware), this script:
  1. Runs PCA on the sample x gene matrix (global, all 155 samples).
  2. Emits a scree plot (variance explained + cumulative).
  3. Emits PC1 vs PC2 scatter panels coloured by condition, cell_type, sex.
  4. Computes robust Mahalanobis distances WITHIN EACH cell_type, fitting
     MinCovDet separately per group on the first K global PCs.
  5. Flags outliers using a chi-squared threshold at alpha = 0.025 *within
     the sample's own cell_type stratum*.
  6. Draws one 97.5% elliptic envelope per cell_type on PC1-PC2.
  7. Writes outlier list and full PC scores to CSV.
"""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Ellipse
from scipy.stats import chi2
from sklearn.covariance import MinCovDet
from sklearn.decomposition import PCA

# ----------------------------- Configuration --------------------------------
PIPELINE_DIR = Path(r"C:\Users\hafsa\Python PD Project\MI_BaggedLASSO Pipeline")
PDHC_DIR     = PIPELINE_DIR / "PDHC new"
OUTPUT_DIR   = PDHC_DIR / "Outputs" / "PCA"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

METADATA_PATH = PIPELINE_DIR / "PDvsHC" / "Meta_data_PDHC.csv"

DATASETS = [
    {"label": "class",   "vst_path": PDHC_DIR / "Outputs" / "VST" / "vst_class_aware_PDHC.csv"},
    {"label": "stratum", "vst_path": PDHC_DIR / "Outputs" / "VST" / "vst_stratum_aware_PDHC.csv"},
]

ALPHA = 0.025                # outlier tail probability for chi^2 threshold
MAX_K = 10                   # cap on number of PCs used for Mahalanobis
TARGET_CUM_VAR = 0.80        # pick K so cumulative variance reaches this
MIN_N_PER_STRATUM = 10       # below this, no per-stratum MCD is attempted
STRATUM_COL = "cell_type"    # column over which to stratify outlier detection

META_COLS = ["condition", "cell_type", "sex"]

In [6]:
# ------------------------------- Helpers ------------------------------------
def load_vst(path: Path) -> pd.DataFrame:
    """Load VST CSV (gene as first column, samples as remaining columns)."""
    df = pd.read_csv(path, index_col=0)
    if df.shape[0] == 155 and df.shape[1] != 155:
        df = df.T
    return df  # genes x samples


def align_metadata(meta: pd.DataFrame, sample_ids) -> pd.DataFrame:
    meta = meta.copy()
    if "sample_id" in meta.columns:
        meta = meta.set_index("sample_id")
    if "condition" not in meta.columns and "Class_Label" in meta.columns:
        meta = meta.rename(columns={"Class_Label": "condition"})
    missing = [c for c in META_COLS if c not in meta.columns]
    if missing:
        raise KeyError(f"Metadata is missing required columns: {missing}")
    return meta.loc[sample_ids, META_COLS]


def pick_K(cum_var: np.ndarray) -> int:
    """First K such that cum_var[K-1] >= TARGET_CUM_VAR, in [3, MAX_K]."""
    K = int(np.searchsorted(cum_var, TARGET_CUM_VAR) + 1)
    return max(3, min(MAX_K, K))


def detect_outliers_per_stratum(scores, sample_ids, stratum_labels,
                                K_global, alpha=ALPHA, min_n=MIN_N_PER_STRATUM):
    """Per-stratum robust Mahalanobis distances + chi^2 outlier flag."""
    sample_ids = list(sample_ids)
    out = pd.DataFrame(
        index=pd.Index(sample_ids, name="sample_id"),
        data={
            "stratum": list(stratum_labels),
            "n_in_stratum": 0,
            "local_K": 0,
            "mahal_d2": np.nan,
            "threshold": np.nan,
            "outlier": False,
            "note": "",
        },
    )
    mcds_2d = {}

    s = pd.Series(stratum_labels, index=sample_ids, name="stratum")
    for stratum_name, group_idx in s.groupby(s).groups.items():
        ids = list(group_idx)
        pos = [sample_ids.index(i) for i in ids]
        n = len(pos)
        out.loc[ids, "n_in_stratum"] = n

        if n < min_n:
            out.loc[ids, "note"] = f"stratum n<{min_n}, skipped"
            continue

        local_K = max(2, min(K_global, n // 4))
        if local_K < 2:
            out.loc[ids, "note"] = "local_K<2, skipped"
            continue

        try:
            mcd = MinCovDet(random_state=0).fit(scores[pos, :local_K])
            d2 = mcd.mahalanobis(scores[pos, :local_K])
        except Exception as exc:
            out.loc[ids, "note"] = f"MCD fit failed: {exc}"
            continue

        thresh = chi2.ppf(1 - alpha, df=local_K)
        out.loc[ids, "local_K"] = local_K
        out.loc[ids, "mahal_d2"] = d2
        out.loc[ids, "threshold"] = thresh
        out.loc[ids, "outlier"] = d2 > thresh

        try:
            mcds_2d[stratum_name] = MinCovDet(random_state=0).fit(scores[pos, :2])
        except Exception:
            pass

    return out, mcds_2d

In [7]:
# ------------------------------- Plots --------------------------------------
def plot_scree(label, var_exp, cum_var, K, out_path):
    n = min(20, len(var_exp))
    x = np.arange(1, n + 1)
    fig, ax1 = plt.subplots(figsize=(9, 5))
    ax1.bar(x, var_exp[:n] * 100, color="#4C8DAE", alpha=0.85)
    ax1.set_xlabel("Principal component")
    ax1.set_ylabel("Variance explained (%)", color="#1F4E79")
    ax1.tick_params(axis="y", labelcolor="#1F4E79")
    ax1.set_xticks(x)

    ax2 = ax1.twinx()
    ax2.plot(x, cum_var[:n] * 100, "o-", color="#B85450")
    ax2.set_ylabel("Cumulative variance (%)", color="#7B2D26")
    ax2.tick_params(axis="y", labelcolor="#7B2D26")
    ax2.axvline(K, ls="--", color="gray", alpha=0.7)
    ax2.text(K + 0.2, 55, f"K={K} for Mahalanobis", color="gray", fontsize=9)

    plt.title(f"Scree plot ({label}-aware) - PD vs HC")
    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.close()


def plot_pc12_panels(label, scores_df, out_path):
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    for ax, var in zip(axes, META_COLS):
        for name, grp in scores_df.groupby(var):
            ax.scatter(grp["PC1"], grp["PC2"], label=str(name),
                       alpha=0.8, s=45, edgecolor="white", linewidth=0.5)
        ax.set_xlabel("PC1")
        ax.set_ylabel("PC2")
        ax.set_title(f"PC1 vs PC2 - {var} ({label}-aware, PD vs HC)")
        ax.legend(fontsize=8, loc="best", frameon=True)
        ax.grid(alpha=0.2)
    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.close()


def _ellipse_from_mcd(mcd, threshold_2d, **kwargs):
    center = mcd.location_
    cov = mcd.covariance_
    vals, vecs = np.linalg.eigh(cov)
    order = vals.argsort()[::-1]
    vals, vecs = vals[order], vecs[:, order]
    angle = np.degrees(np.arctan2(vecs[1, 0], vecs[0, 0]))
    width, height = 2 * np.sqrt(vals * threshold_2d)
    return Ellipse(xy=center, width=width, height=height, angle=angle, **kwargs)


def plot_envelope_per_stratum(label, scores_df, mcds_2d, threshold_2d, out_path):
    fig, ax = plt.subplots(figsize=(10, 8))
    strata = sorted(scores_df[STRATUM_COL].unique())
    cmap = plt.get_cmap("tab10")

    for i, stratum in enumerate(strata):
        color = cmap(i % 10)
        sub = scores_df[scores_df[STRATUM_COL] == stratum]
        inliers = sub[~sub["outlier"]]
        outliers = sub[sub["outlier"]]
        ax.scatter(inliers["PC1"], inliers["PC2"],
                   c=[color], s=42, alpha=0.75,
                   edgecolor="white", linewidth=0.4,
                   label=f"{stratum} (n={len(sub)})")
        ax.scatter(outliers["PC1"], outliers["PC2"],
                   c=[color], s=110, edgecolor="black", linewidth=1.2, marker="X")

        if stratum in mcds_2d:
            ell = _ellipse_from_mcd(
                mcds_2d[stratum], threshold_2d,
                edgecolor=color, facecolor="none",
                linewidth=2, linestyle="--",
            )
            ax.add_patch(ell)

    out_all = scores_df[scores_df["outlier"]]
    for sid, row in out_all.iterrows():
        ax.annotate(sid, (row["PC1"], row["PC2"]),
                    fontsize=7, ha="left", va="bottom")

    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
    ax.set_title(
        f"Per-{STRATUM_COL} elliptic envelopes on PC1-PC2 ({label}-aware, PD vs HC)\n"
        f"X marker = outlier within its {STRATUM_COL}"
    )
    ax.legend(loc="best", fontsize=9)
    ax.grid(alpha=0.2)
    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.close()


def plot_mahalanobis_per_stratum(label, scores_df, out_path):
    strata = sorted(scores_df[STRATUM_COL].unique())
    n = len(strata)
    cols = min(3, n)
    rows = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(7 * cols, 4 * rows), squeeze=False)

    for ax_idx, stratum in enumerate(strata):
        ax = axes[ax_idx // cols][ax_idx % cols]
        sub = scores_df[scores_df[STRATUM_COL] == stratum].copy()

        if sub["mahal_d2"].isna().all():
            ax.text(0.5, 0.5,
                    f"{stratum}\nstratum too small (n={len(sub)} < {MIN_N_PER_STRATUM})",
                    ha="center", va="center", transform=ax.transAxes)
            ax.set_xticks([]); ax.set_yticks([])
            continue

        sub = sub.sort_values("mahal_d2")
        threshold = sub["threshold"].iloc[0]
        colors = ["#B85450" if v > threshold else "#4C8DAE"
                  for v in sub["mahal_d2"].values]
        ax.bar(range(len(sub)), sub["mahal_d2"].values, color=colors)
        ax.axhline(threshold, ls="--", color="black",
                   label=f"chi^2 thresh = {threshold:.2f}")
        ax.set_title(f"{stratum} (n={len(sub)}, K={int(sub['local_K'].iloc[0])})")
        ax.set_xlabel("Sample (sorted)")
        ax.set_ylabel("Squared Mahalanobis distance")
        ax.legend(fontsize=8)

    for ax_idx in range(len(strata), rows * cols):
        axes[ax_idx // cols][ax_idx % cols].axis("off")

    fig.suptitle(
        f"Per-{STRATUM_COL} robust Mahalanobis distances ({label}-aware, PD vs HC)",
        fontsize=13,
    )
    plt.tight_layout(rect=(0, 0, 1, 0.96))
    plt.savefig(out_path, dpi=200)
    plt.close()

In [8]:
# ----------------------------- Main pipeline --------------------------------
def run_pca_pipeline(label: str, vst_path: Path, meta_full: pd.DataFrame):
    if not vst_path.exists():
        print(f"[{label}] skip - file not found: {vst_path}")
        return

    print(f"\n========== {label}-aware ==========")
    vst = load_vst(vst_path)
    sample_ids = vst.columns.tolist()
    X = vst.T.values
    print(f"Matrix       : {X.shape[0]} samples x {X.shape[1]} genes")

    meta = align_metadata(meta_full, sample_ids)

    n_components = min(50, X.shape[0] - 1)
    pca = PCA(n_components=n_components, random_state=0)
    scores = pca.fit_transform(X)
    var_exp = pca.explained_variance_ratio_
    cum_var = var_exp.cumsum()
    K_global = pick_K(cum_var)
    print(f"PCs (global K): K={K_global}  (cum. var = {cum_var[K_global-1]:.1%})")

    pc_cols = [f"PC{i + 1}" for i in range(n_components)]
    scores_df = pd.DataFrame(scores, index=sample_ids, columns=pc_cols)
    scores_df = pd.concat([scores_df, meta], axis=1)

    out_table, mcds_2d = detect_outliers_per_stratum(
        scores=scores,
        sample_ids=sample_ids,
        stratum_labels=meta[STRATUM_COL].values,
        K_global=K_global,
        alpha=ALPHA,
        min_n=MIN_N_PER_STRATUM,
    )
    scores_df["mahal_d2"]   = out_table["mahal_d2"]
    scores_df["threshold"]  = out_table["threshold"]
    scores_df["local_K"]    = out_table["local_K"]
    scores_df["n_in_stratum"] = out_table["n_in_stratum"]
    scores_df["outlier"]    = out_table["outlier"].astype(bool)
    scores_df["note"]       = out_table["note"]

    n_total = len(scores_df)
    n_out   = int(scores_df["outlier"].sum())
    n_skipped_strata = scores_df["note"].str.contains("skipped").sum()
    print(f"Per-{STRATUM_COL} stats:")
    summary = (out_table.groupby("stratum")
               .agg(n=("n_in_stratum", "first"),
                    K=("local_K", "first"),
                    threshold=("threshold", "first"),
                    n_outliers=("outlier", "sum")))
    print(summary.to_string())
    print(f"Total outliers: {n_out} / {n_total} "
          f"({100 * n_out / n_total:.1f}%, alpha={ALPHA})")
    if n_skipped_strata:
        print(f"Samples skipped (stratum too small): {int(n_skipped_strata)}")

    threshold_2d = chi2.ppf(1 - ALPHA, df=2)

    plot_scree(label, var_exp, cum_var, K_global,
               OUTPUT_DIR / f"pca_{label}_scree_PDHC.png")
    plot_pc12_panels(label, scores_df,
                     OUTPUT_DIR / f"pca_{label}_pc12_byvar_PDHC.png")
    plot_envelope_per_stratum(label, scores_df, mcds_2d, threshold_2d,
                              OUTPUT_DIR / f"pca_{label}_envelope_per_stratum_PDHC.png")
    plot_mahalanobis_per_stratum(label, scores_df,
                                 OUTPUT_DIR / f"pca_{label}_mahalanobis_per_stratum_PDHC.png")

    scores_df.to_csv(OUTPUT_DIR / f"pca_{label}_scores_PDHC.csv")

    out_csv = OUTPUT_DIR / f"pca_{label}_outliers_PDHC.csv"
    if n_out > 0:
        outlier_table = (
            scores_df[scores_df["outlier"]]
            .sort_values(["cell_type", "mahal_d2"], ascending=[True, False])
            [META_COLS + ["n_in_stratum", "local_K", "mahal_d2", "threshold"]
             + [f"PC{i + 1}" for i in range(K_global)]]
        )
        outlier_table.to_csv(out_csv)
        print(f"\nFlagged outliers ({label}-aware), grouped by {STRATUM_COL}:")
        print(outlier_table.to_string())
    else:
        cols = (META_COLS + ["n_in_stratum", "local_K", "mahal_d2", "threshold"]
                + [f"PC{i + 1}" for i in range(K_global)])
        pd.DataFrame(columns=cols).to_csv(out_csv)
        print(f"No outliers flagged at alpha={ALPHA} ({label}-aware).")

    print(f"\nOutputs ({label}-aware) in: {OUTPUT_DIR}")

In [9]:
# ------------------------------- Run ----------------------------------------
if not METADATA_PATH.exists():
    raise FileNotFoundError(f"Metadata file not found: {METADATA_PATH}")
meta_full = pd.read_csv(METADATA_PATH)
for ds in DATASETS:
    run_pca_pipeline(ds["label"], ds["vst_path"], meta_full)
print(f"\nAll outputs in: {OUTPUT_DIR}")


========== class-aware ==========
Matrix       : 155 samples x 12103 genes
PCs (global K): K=10  (cum. var = 51.5%)
Per-cell_type stats:
          n   K  threshold  n_outliers
stratum                               
CD4      53  10  20.483177          21
CD8      56  10  20.483177          22
PBMC     46  10  20.483177          15
Total outliers: 58 / 155 (37.4%, alpha=0.025)

Flagged outliers (class-aware), grouped by cell_type:
                   condition cell_type sex  n_in_stratum  local_K      mahal_d2  threshold        PC1        PC2        PC3        PC4        PC5        PC6        PC7        PC8        PC9       PC10
ES_CD4_M_2763             PD       CD4   M            53       10  24512.741672  20.483177   5.184887  19.449754 -84.130052  45.721977  -2.638720   5.869400  95.951003   7.018316 -17.915765  81.847248
HC_CD4_F_2791             HC       CD4   F            53       10    647.049535  20.483177 -20.549840 -23.287602 -17.568589  -6.403472 -36.233296  19.301863  -7.705